# LIBERO Simulation — Lifelong Robot Learning Benchmark

This notebook sets up the **LIBERO** benchmark for headless simulation in Google Colab.

LIBERO provides 130 manipulation tasks across 4 suites:

| Suite | Tasks | Tests |
|---|---|---|
| `libero_spatial` | 10 | Spatial relationship transfer |
| `libero_object` | 10 | Object knowledge transfer |
| `libero_goal` | 10 | Goal/activity transfer |
| `libero_100` | 100 | Large-scale entangled transfer |

**Requirements:** Select a **GPU runtime** (Runtime → Change runtime type → T4 GPU).

> **Note:** LIBERO uses `robosuite==1.4.0` (not v1.5). The notebooks are kept separate to avoid version conflicts.

## 1. System Setup & EGL Rendering

In [ ]:
%%bash
# System dependencies for headless MuJoCo rendering
apt-get update -qq
apt-get install -y -qq libegl1-mesa-dev libgl1-mesa-glx libosmesa6-dev libglfw3 ffmpeg patchelf > /dev/null 2>&1

# NVIDIA EGL ICD config
mkdir -p /usr/share/glvnd/egl_vendor.d
cat > /usr/share/glvnd/egl_vendor.d/10_nvidia.json << 'EOF'
{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}
EOF

echo "System dependencies installed."

In [ ]:
import os

# MUST be set BEFORE importing mujoco or robosuite
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"

# Fix the "No private macro file found" warning
import robosuite
macros_private = os.path.join(os.path.dirname(robosuite.__file__), "macros_private.py")
if not os.path.exists(macros_private):
    with open(macros_private, "w") as f:
        f.write("# Auto-generated private macros\n")

print(f"robosuite {robosuite.__version__} ready (EGL rendering)")

# Verify LIBERO
from libero.libero import benchmark
print(f"LIBERO suites: {list(benchmark.get_benchmark_dict().keys())}")

## 2. Install LIBERO and Dependencies

LIBERO pins robosuite 1.4.x and specific dependency versions. We install compatible versions below.

**Important:** This cell restarts the runtime after install to fix a numpy binary incompatibility with Colab's Python 3.12. After the restart, **skip this cell** and continue from Cell 3 (environment variables).

In [ ]:
%%bash
# Install LIBERO-compatible dependencies
pip install -q robosuite==1.4.1 robomimic==0.2.0 bddl==1.0.1
pip install -q hydra-core easydict einops cloudpickle "gym==0.25.2" imageio[ffmpeg] matplotlib

# Clone and install LIBERO
if [ ! -d "LIBERO" ]; then
    git clone --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git
fi
pip install -q -e LIBERO/

# Pin numpy: >=2.0 (Colab prebuilt packages) and <2.1 (numba compat)
pip install -q "numpy>=2.0,<2.1"

echo "LIBERO installation complete. Restarting runtime..."

In [ ]:
# Restart runtime to clear stale numpy bindings
# After restart, skip to Cell 3 (environment variables)
import os
os.kill(os.getpid(), 9)

## 3. Helper Functions

In [ ]:
import numpy as np
import imageio
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import base64

from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv


def get_task_env(suite_name, task_id, res=128):
    """Create a LIBERO environment for a specific task.

    Args:
        suite_name: one of 'libero_spatial', 'libero_object', 'libero_goal',
                    'libero_10', 'libero_90'
        task_id: integer index of the task within the suite
        res: camera resolution (height and width)

    Returns:
        env: the OffScreenRenderEnv
        task: the task object (has .name, .language, etc.)
        task_suite: the suite object (for getting init states)
    """
    benchmark_dict = benchmark.get_benchmark_dict()
    task_suite = benchmark_dict[suite_name]()
    task = task_suite.get_task(task_id)

    bddl_file = os.path.join(
        get_libero_path("bddl_files"),
        task.problem_folder,
        task.bddl_file,
    )

    env = OffScreenRenderEnv(
        bddl_file_name=bddl_file,
        camera_heights=res,
        camera_widths=res,
    )
    env.seed(0)
    env.reset()

    # Set to a fixed initial state for reproducibility
    init_states = task_suite.get_task_init_states(task_id)
    if len(init_states) > 0:
        env.set_init_state(init_states[0])

    return env, task, task_suite


def run_libero_rollout(env, policy_fn=None, max_steps=300, camera="agentview"):
    """Run a rollout in a LIBERO environment.

    Args:
        env: LIBERO OffScreenRenderEnv
        policy_fn: callable(obs) -> action (7D). If None, uses zero actions.
        max_steps: maximum number of steps
        camera: camera name for frame capture

    Returns:
        frames: list of RGB frames
        rewards: list of per-step rewards
        success: whether the task was completed
    """
    obs = env.reset()
    frames = []
    rewards = []
    success = False

    for step in range(max_steps):
        if policy_fn is not None:
            action = policy_fn(obs)
        else:
            action = np.zeros(7)  # no-op: stay still

        obs, reward, done, info = env.step(action)
        rewards.append(reward)

        cam_key = f"{camera}_image"
        if cam_key in obs:
            frame = np.flip(obs[cam_key], axis=0)
            frames.append(frame)

        if done:
            success = True
            break

    return frames, rewards, success


def save_video(frames, path="rollout.mp4", fps=20):
    """Save frames as MP4 video."""
    writer = imageio.get_writer(path, fps=fps)
    for frame in frames:
        writer.append_data(frame)
    writer.close()
    print(f"Video saved: {path} ({len(frames)} frames)")


def show_video(path):
    """Display MP4 inline in Colab."""
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<video controls width="512"><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'
    ))


print("Helper functions loaded.")

---
## 4. Explore Available Tasks

List all tasks in each LIBERO suite with their natural language descriptions.

In [ ]:
benchmark_dict = benchmark.get_benchmark_dict()

for suite_name in ["libero_spatial", "libero_object", "libero_goal"]:
    suite_obj = benchmark_dict[suite_name]()
    print(f"\n{'='*60}")
    print(f"Suite: {suite_name} ({suite_obj.n_tasks} tasks)")
    print(f"{'='*60}")
    for i in range(suite_obj.n_tasks):
        task = suite_obj.get_task(i)
        print(f"  [{i}] {task.language}")

---
## 5. Run a LIBERO-Spatial Task

LIBERO-Spatial tests spatial relationship understanding — same objects, different arrangements.

In [ ]:
env, task, suite_obj = get_task_env("libero_spatial", task_id=0)

print(f"Task: {task.language}")
print(f"BDDL file: {task.bddl_file}")

# Run with random actions
def random_policy(obs):
    return np.random.uniform(-0.3, 0.3, size=7)

frames, rewards, success = run_libero_rollout(env, policy_fn=random_policy, max_steps=200)
save_video(frames, "libero_spatial_0.mp4")
show_video("libero_spatial_0.mp4")

print(f"\nTask completed: {success}")
print(f"Total reward: {sum(rewards):.4f}")
env.close()

---
## 6. Run a LIBERO-Object Task

LIBERO-Object tests object knowledge transfer — same spatial layout, different objects.

In [ ]:
env, task, _ = get_task_env("libero_object", task_id=0)

print(f"Task: {task.language}")

frames, rewards, success = run_libero_rollout(env, policy_fn=random_policy, max_steps=200)
save_video(frames, "libero_object_0.mp4")
show_video("libero_object_0.mp4")

print(f"\nTask completed: {success}")
print(f"Total reward: {sum(rewards):.4f}")
env.close()

---
## 7. Run a LIBERO-Goal Task

LIBERO-Goal tests goal knowledge transfer — same scene, different objectives.

In [ ]:
env, task, _ = get_task_env("libero_goal", task_id=0)

print(f"Task: {task.language}")

frames, rewards, success = run_libero_rollout(env, policy_fn=random_policy, max_steps=200)
save_video(frames, "libero_goal_0.mp4")
show_video("libero_goal_0.mp4")

print(f"\nTask completed: {success}")
print(f"Total reward: {sum(rewards):.4f}")
env.close()

---
## 8. Batch Evaluation: Run All Tasks in a Suite

Run all 10 tasks in a suite with random actions and report results.

In [ ]:
suite_name = "libero_spatial"  # Change to libero_object or libero_goal
benchmark_dict = benchmark.get_benchmark_dict()
suite_obj = benchmark_dict[suite_name]()

results = []

for task_id in range(suite_obj.n_tasks):
    env, task, _ = get_task_env(suite_name, task_id, res=64)  # low res for speed
    frames, rewards, success = run_libero_rollout(
        env, policy_fn=random_policy, max_steps=100
    )
    total_r = sum(rewards)
    results.append({"id": task_id, "task": task.language, "reward": total_r, "success": success})
    print(f"  [{task_id}] reward={total_r:.3f}  success={success}  — {task.language}")
    env.close()

print(f"\nSuccess rate: {sum(r['success'] for r in results)}/{len(results)}")

---
## 9. Observation Space Explorer

See what observations LIBERO tasks provide.

In [ ]:
env, task, _ = get_task_env("libero_spatial", task_id=0)
obs = env.reset()

print(f"Task: {task.language}")
print(f"\nObservation keys and shapes:")
print("-" * 55)
for key in sorted(obs.keys()):
    val = obs[key]
    if isinstance(val, np.ndarray):
        print(f"  {key:35s}  shape={str(val.shape):15s}  dtype={val.dtype}")
    else:
        print(f"  {key:35s}  type={type(val).__name__}")

env.close()

---
## 10. Save Results to Google Drive (Optional)

Mount Google Drive and save videos/checkpoints to persist across sessions.

In [ ]:
# Uncomment to mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# import shutil
# save_dir = '/content/drive/MyDrive/AUTOLAB/libero_results'
# os.makedirs(save_dir, exist_ok=True)
# shutil.copy('libero_spatial_0.mp4', save_dir)
# print(f'Saved to {save_dir}')

---
## LIBERO Training Reference

To train a policy on LIBERO benchmarks (requires demonstration datasets):

```bash
# Download demonstration datasets
python LIBERO/benchmark_scripts/download_libero_datasets.py --datasets libero_spatial

# Train a BC-RNN policy on LIBERO-Spatial
python LIBERO/libero/lifelong/main.py \
    seed=42 \
    benchmark_name=LIBERO_SPATIAL \
    policy=bc_rnn_policy \
    lifelong=multitask
```

**Available policies:** `bc_rnn_policy`, `bc_transformer_policy`, `bc_vilt_policy`

**Lifelong learning methods:** `base` (sequential), `er` (experience replay), `ewc` (elastic weight consolidation), `packnet`, `multitask` (joint training)